# 05 — Interactive demo on Colab (Gradio)

Type an instruction, watch the robot act in LIBERO: the same demo as `.\demo` on the laptop
(`groundedvla/demo/`), with a Gradio page instead.

* **GPU:** T4 is enough (`Runtime → Change runtime type → T4 GPU`).
* **Weights:** read from `MyDrive/groundedvla/models/` (downloaded there once if missing), copied to local disk for speed.
* **Link:** the launch cell prints a public `https://….gradio.live` link. Open it in any browser, which also works when
  this notebook runs from VS Code. Anyone with the link can use the demo while it runs, so don't post it publicly.
  Run `ui.close()` (last cell) when done.
* The launch cell returns immediately; the demo keeps running in the background, so the kernel stays free.

In [ ]:
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"
BRANCH = "main"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

In [ ]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
%cd /content/GroundedVLA
!bash scripts/colab_install.sh requirements.txt
# gradio 6 (preinstalled on Colab) needs a newer huggingface-hub than LeRobot allows, so use gradio 5
!pip install -q "gradio>=5.20,<6" "huggingface-hub==0.35.3"

In [ ]:
from pathlib import Path
from groundedvla.utils.colab import setup_session, stage_to_local
from groundedvla.policies.registry import load_registry, resolve_weights
paths = setup_session()          # env vars (EGL, caches), Drive folders, LIBERO config

# weights: Drive copy (downloaded to Drive the first time), then a fast local copy
for name in ["smolvla_libero"]:
    spec = load_registry()[name]
    print(name, "on Drive:", resolve_weights(spec["hub_id"], spec["folder"], paths))
    stage_to_local(f"models/{spec['folder']}")

## Launch

`DEMO_ARGS` takes the same arguments as the laptop demo:

| What | `DEMO_ARGS` |
|---|---|
| Hugging Face's LIBERO SmolVLA | `smolvla_libero --task T3` |
| one of our checkpoints (on Drive) | `smolvla_ckpt --checkpoint /content/drive/MyDrive/groundedvla/checkpoints/v1_rgb/step_020000 --horizon 10 --label V1` |
| two models side by side | add `--compare smolvla_ckpt --compare-checkpoint <dir> --compare-label V3` |

Loading takes about 1–2 minutes (the page says *loading* until then). A policy call takes roughly 0.5 s on a T4 (1–1.4 s on the laptop RTX 3050);
`smolvla_libero` re-plans every step, so leave **Smooth playback** on: the run is computed first, with a progress bar, then played at 20 Hz.

In [ ]:
from groundedvla.demo.gradio_app import launch
DEMO_ARGS = "smolvla_libero --task T3"
ui = launch(DEMO_ARGS)           # prints the public link; run again to restart with other arguments

In [ ]:
# ui.close()                     # stop the demo and free the GPU